In [ ]:
!pip install -q diffusers transformers accelerate safetensors

import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
)
pipe = pipe.to("cuda")

print("Modelo cargado correctamente.")

In [ ]:
# Sube english_cards_300_words_template.json (ya con las 400 tarjetas,
# incluida la categoría "Programming") a /content antes de correr esto.
import json

with open('/content/english_cards_300_words_template.json', encoding='utf-8') as f:
    cards = json.load(f)

programming_cards = [c for c in cards if c["category"] == "Programming"]

STYLE_SUFFIX = "flat vector illustration, simple shapes, soft colors, white background, no text, no words, centered, digital art"

def build_prompt(card):
    word = card["word"]
    # Los términos de programación mezclan conceptos abstractos (loop,
    # recursion, boolean) con elementos más concretos (server, database,
    # terminal); se usa siempre la variante "concepto" en vez de la de
    # "icono de categoría" para que funcione bien en ambos casos.
    return f"a simple flat illustration representing the concept of '{word}' in software programming, {STYLE_SUFFIX}"

NEGATIVE_PROMPT = "text, letters, watermark, signature, blurry, photo, realistic, low quality"

prompts = []
for card in programming_cards:
    prompts.append({
        "id": card["id"],
        "word": card["word"],
        "filename": card["image"],
        "prompt": build_prompt(card),
        "negative_prompt": NEGATIVE_PROMPT,
    })

print(f"Total prompts generados: {len(prompts)}")

In [ ]:
import os

output_dir = "/content/images"
os.makedirs(output_dir, exist_ok=True)

test_card = prompts[0]
print("Palabra:", test_card["word"])
print("Prompt:", test_card["prompt"])

image = pipe(
    prompt=test_card["prompt"],
    num_inference_steps=4,
    guidance_scale=0.0,
).images[0]

image.save(f"{output_dir}/{test_card['filename']}")
image

In [ ]:
from tqdm.auto import tqdm
import os

output_dir = "/content/images"
os.makedirs(output_dir, exist_ok=True)

errors = []

for card in tqdm(prompts, desc="Generando imágenes"):
    filepath = os.path.join(output_dir, card["filename"])

    if os.path.exists(filepath):
        continue  # ya generada antes, la salta (por si la sesión se reinició)

    try:
        image = pipe(
            prompt=card["prompt"],
            num_inference_steps=4,
            guidance_scale=0.0,
        ).images[0]
        image.save(filepath)
    except Exception as e:
        errors.append((card["word"], str(e)))

total_generadas = len(os.listdir(output_dir))
print(f"\nListo. Imágenes en la carpeta: {total_generadas} de {len(prompts)}")

if errors:
    print(f"\n{len(errors)} palabras fallaron:")
    for word, err in errors:
        print(f"- {word}: {err}")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/programming_flashcard_images", "zip", output_dir)
files.download("/content/programming_flashcard_images.zip")